# INT8 Dynamic Quantisation Experiment

Applies post-training INT8 dynamic quantisation to the trained seed-2026 model and compares the FP32 baseline against the quantised model on model size, single-thread CPU latency, and external sensitivity / specificity.

In [ ]:
# ------------------------------------------------------------
# Configuration
# Set these paths to your local or Drive dataset locations.
# ------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')

import os

DATA_ROOT = "PATH/TO/PROJECT"

BRATS_SOURCE_DIR = os.path.join(DATA_ROOT, "brats_africa/glioma")
NIGERIAN_HEALTHY = os.path.join(DATA_ROOT, "nigerian_healthy/axial")
EXP_TEST_DIR     = os.path.join(DATA_ROOT, "brats_experiment/Testing")
DINOV2_WEIGHTS   = os.path.join(DATA_ROOT, "models/dinov2_vits14_weights.pth")
SEED_2026_CKPT   = os.path.join(DATA_ROOT, "models/dinov2_brats_augmented_seed2026.pth")
QUANT_EXP_DIR    = os.path.join(DATA_ROOT, "quantization_experiment")
os.makedirs(QUANT_EXP_DIR, exist_ok=True)

SEED              = 2026
BRATS_TRAIN_SPLIT = 0.60
IMG_SIZE          = 224
BATCH_SIZE        = 32

In [ ]:
# Quantization Experiment — Phase 1: FP32 Baseline
# Model: DINOv2 ViT-S/14 + trained head (seed 2026)
# All measurements on CPU for parity with Phase 3.

import os
import json
import time
import platform
import random
import numpy as np
import torch
import torch.nn as nn
import cv2

# --- FORCE CPU. Do not fall back to cuda. ---
device = torch.device('cpu')
torch.set_num_threads(1)  # single-threaded for reproducibility
print(f"Device: {device}  |  Threads: {torch.get_num_threads()}")
print(f"CPU: {platform.processor() or 'unknown'}  |  Cores available: {os.cpu_count()}")

# --- Read-only paths ---
os.makedirs(QUANT_EXP_DIR, exist_ok=True)
BASELINE_JSON      = os.path.join(QUANT_EXP_DIR, "baseline_fp32.json")
BASELINE_WEIGHTS   = os.path.join(QUANT_EXP_DIR, "fp32_state_dict.pth")

In [ ]:
# Load DINOv2 backbone + trained head onto CPU.

print(" Loading DINOv2 backbone weights  CPU...")
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=False)
backbone.load_state_dict(torch.load(DINOV2_WEIGHTS, map_location='cpu'))
for p in backbone.parameters():
    p.requires_grad = False

class DINOv2Classifier(nn.Module):
    """Identical architecture to training notebook."""def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.head(features)

model = DINOv2Classifier(backbone).to(device)
ckpt  = torch.load(SEED_2026_CKPT, map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Model loaded (checkpoint val_acc: {ckpt['val_acc']:.4f}, epoch {ckpt['epoch']+1})")

In [ ]:
assert next(model.parameters()).device.type == 'cpu', "Model is not on CPU!"
print(f"Model confirmed on: {next(model.parameters()).device}")

In [ ]:
# Two size numbers matter:
# 1. On-disk state_dict size (what you actually ship)
# 2. Parameter count breakdown (backbone vs head, for reporting)

# --- Save state_dict to a tmp file to get honest disk size ---
torch.save(model.state_dict(), BASELINE_WEIGHTS)
disk_size_bytes = os.path.getsize(BASELINE_WEIGHTS)
disk_size_mb    = disk_size_bytes / (1024 * 1024)

# --- Parameter counts ---
backbone_params = sum(p.numel() for p in model.backbone.parameters())
head_params     = sum(p.numel() for p in model.head.parameters())
total_params    = backbone_params + head_params

# Theoretical size if all FP32 (4 bytes/param)
theoretical_mb  = (total_params * 4) / (1024 * 1024)

print("=" * 60)
print("FP32 MODEL SIZE")
print("=" * 60)
print(f"Backbone params (frozen)  : {backbone_params:>12,}")
print(f"Head params (trained)     : {head_params:>12,}")
print(f"Total params              : {total_params:>12,}")
print(f"Theoretical FP32 size     : {theoretical_mb:>12.2f} MB  (params × 4 bytes)")
print(f"Actual state_dict on disk : {disk_size_mb:>12.2f} MB")
print("=" * 60)

In [ ]:
# Latency: single-image inference, single-threaded.
# This is the primitive that matters — per-slice inference cost.
# For a full MRI volume, multiply by slice count.

N_WARMUP = 20
N_RUNS   = 100

# Realistic input: preprocessed tensor at inference resolution
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)

# --- Warmup (JIT, cache warmup, etc.) ---
print(f"Warmup: {N_WARMUP} runs...")
with torch.no_grad():
    for _ in range(N_WARMUP):
        _ = model(dummy_input)

# --- Timed runs ---
print(f"Measuring: {N_RUNS} runs...")
latencies_ms = []
with torch.no_grad():
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        _ = model(dummy_input)
        t1 = time.perf_counter()
        latencies_ms.append((t1 - t0) * 1000)

latencies_ms = np.array(latencies_ms)

print("\n" + "=" * 60)
print("FP32 CPU LATENCY (batch=1, single-threaded)")
print("=" * 60)
print(f"Mean    : {latencies_ms.mean():>8.2f} ms")
print(f"Median  : {np.median(latencies_ms):>8.2f} ms")
print(f"Std     : {latencies_ms.std():>8.2f} ms")
print(f"p95     : {np.percentile(latencies_ms, 95):>8.2f} ms")
print(f"p99     : {np.percentile(latencies_ms, 99):>8.2f} ms")
print(f"Min     : {latencies_ms.min():>8.2f} ms")
print(f"Max     : {latencies_ms.max():>8.2f} ms")
print("=" * 60)

In [ ]:
# Rebuild seed 2026's BraTS holdout in memory (same trick as before)
# and run sensitivity + specificity on CPU.

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def preprocess(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype("float32") / 255.0
    img = (img - MEAN) / STD
    return torch.tensor(img).permute(2, 0, 1).unsqueeze(0).float().to(device)

def extract_patient_id(filename):
    return filename.split('_slice')[0]

def get_holdout_for_seed(brats_source, seed, train_split=0.60):
    random.seed(seed)
    files = sorted([f for f in os.listdir(brats_source)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    p2s = {}
    for f in files:
        p2s.setdefault(extract_patient_id(f), []).append(f)
    patients = sorted(p2s.keys())
    random.shuffle(patients)
    split = int(len(patients) * train_split)
    holdout_patients = patients[split:]
    return [s for p in holdout_patients for s in p2s[p]]

def eval_metric(model, source_dir, filenames, ground_truth):
    preds = []
    for f in filenames:
        img = cv2.imread(os.path.join(source_dir, f))
        if img is None: continue
        with torch.no_grad():
            preds.append(model(preprocess(img)).item())
    preds = np.array(preds)
    correct = int(np.sum(preds > 0.5)) if ground_truth == 'tumor' else int(np.sum(preds < 0.5))
    return correct / len(preds), correct, len(preds), preds

# --- Sensitivity: BraTS holdout for seed 2026 ---
holdout_files = get_holdout_for_seed(BRATS_SOURCE_DIR, SEED, BRATS_TRAIN_SPLIT)
print(f"BraTS holdout (seed {SEED}): {len(holdout_files)} slices")
sens, s_correct, s_total, _ = eval_metric(model, BRATS_SOURCE_DIR, holdout_files, 'tumor')
print(f"Sensitivity: {sens:.4f}  ({s_correct}/{s_total})")

# --- Specificity: Nigerian healthy ---
healthy_files = sorted([f for f in os.listdir(NIGERIAN_HEALTHY)
                        if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
print(f"\nNigerian healthy: {len(healthy_files)} scans")
spec, h_correct, h_total, _ = eval_metric(model, NIGERIAN_HEALTHY, healthy_files, 'healthy')
print(f"Specificity: {spec:.4f}  ({h_correct}/{h_total})")

print("\n" + "=" * 60)
print("FP32 CLINICAL METRICS (CPU)")
print("=" * 60)
print(f"Sensitivity : {sens:.2%}  ({s_correct}/{s_total})")
print(f"Specificity : {spec:.2%}  ({h_correct}/{h_total})")
print("=" * 60)

In [ ]:
# Persist every baseline number so Phase 3 can compare cleanly.

baseline = {
    'phase'                 : 'FP32 baseline',
    'seed'                  : SEED,
    'source_checkpoint'     : SEED_2026_CKPT,
    'device'                : str(device),
    'num_threads'           : torch.get_num_threads(),
    'cpu_info'              : platform.processor() or 'unknown',
    'cpu_cores_available'   : os.cpu_count(),

    # Size
    'backbone_params'       : backbone_params,
    'head_params'           : head_params,
    'total_params'          : total_params,
    'theoretical_size_mb'   : round(theoretical_mb, 2),
    'disk_state_dict_mb'    : round(disk_size_mb, 2),

    # Latency (single-image, batch=1, single-threaded)
    'latency_batch_size'    : 1,
    'latency_n_warmup'      : N_WARMUP,
    'latency_n_runs'        : N_RUNS,
    'latency_mean_ms'       : round(float(latencies_ms.mean()), 2),
    'latency_median_ms'     : round(float(np.median(latencies_ms)), 2),
    'latency_std_ms'        : round(float(latencies_ms.std()), 2),
    'latency_p95_ms'        : round(float(np.percentile(latencies_ms, 95)), 2),
    'latency_p99_ms'        : round(float(np.percentile(latencies_ms, 99)), 2),

    # Clinical
    'sensitivity'           : round(sens, 4),
    'sensitivity_correct'   : s_correct,
    'sensitivity_total'     : s_total,
    'specificity'           : round(spec, 4),
    'specificity_correct'   : h_correct,
    'specificity_total'     : h_total,
}

with open(BASELINE_JSON, 'w') as f:
    json.dump(baseline, f, indent=2)

print(f"Baseline saved to: {BASELINE_JSON}")
print(f"FP32 state_dict cached at: {BASELINE_WEIGHTS}")
print("\n" + json.dumps(baseline, indent=2))

In [ ]:
# Phase 2: Apply INT8 Dynamic Quantization
# Targets nn.Linear only — Softmax/GELU/LayerNorm stay in FP32.

import torch.quantization

# --- The actual quantization (3 lines) ---
quantized_model = torch.quantization.quantize_dynamic(
    model,                      # the FP32 model we just loaded
    {nn.Linear},                # which layer types to quantize
    dtype=torch.qint8           # target precision
)
quantized_model.eval()

print(" Dynamic quantization applied")
print(f"Targeted layer type: nn.Linear")
print(f"Weight dtype       : torch.qint8")

In [ ]:
# Sanity check: count how many Linear layers actually became
# quantized (they get replaced by DynamicQuantizedLinear).

from torch.nn.quantized.dynamic import Linear as QuantizedLinear

fp32_linears      = 0
quantized_linears = 0
other_layers      = {}

for name, module in quantized_model.named_modules():
    if isinstance(module, QuantizedLinear):
        quantized_linears += 1
    elif isinstance(module, nn.Linear):
        fp32_linears += 1
    else:
        cls = type(module).__name__
        other_layers[cls] = other_layers.get(cls, 0) + 1

print("=" * 60)
print("LAYER-LEVEL QUANTIZATION AUDIT")
print("=" * 60)
print(f"Quantized Linear layers    : {quantized_linears}")
print(f"Remaining FP32 Linear      : {fp32_linears}  (should be 0)")
print(f"\nOther layer types (unchanged, stay FP32):")
for cls, n in sorted(other_layers.items(), key=lambda x: -x[1])[:10]:
    print(f"{cls:<30} : {n}")

assert fp32_linears == 0, "Some Linear layers didn't get quantized!"
print("\n All nn.Linear layers successfully replaced with quantized versions.")

In [ ]:
# Save the quantized model. Note: use torch.jit.save or plain
# torch.save on state_dict for portability.
# We save state_dict for simplicity; loading it requires
# re-running quantize_dynamic on a fresh FP32 model first.

QUANTIZED_WEIGHTS = os.path.join(QUANT_EXP_DIR, "int8_dynamic_state_dict.pth")
QUANTIZED_FULL    = os.path.join(QUANT_EXP_DIR, "int8_dynamic_full_model.pth")

# Save state_dict (standard, small)
torch.save(quantized_model.state_dict(), QUANTIZED_WEIGHTS)

# Also save the full model (easier to reload without re-quantizing)
torch.save(quantized_model, QUANTIZED_FULL)

state_dict_size_mb = os.path.getsize(QUANTIZED_WEIGHTS) / (1024 * 1024)
full_model_size_mb = os.path.getsize(QUANTIZED_FULL) / (1024 * 1024)

print(f"Quantized state_dict : {state_dict_size_mb:.2f} MB")
print(f"Quantized full model : {full_model_size_mb:.2f} MB")
print(f"\nSaved to: {QUANT_EXP_DIR}")

In [ ]:
# Phase 3a: Quantized CPU latency
# Same benchmark protocol as FP32 for direct comparison.

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)

print(f"Warmup: {N_WARMUP} runs...")
with torch.no_grad():
    for _ in range(N_WARMUP):
        _ = quantized_model(dummy_input)

print(f"Measuring: {N_RUNS} runs...")
q_latencies_ms = []
with torch.no_grad():
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        _ = quantized_model(dummy_input)
        t1 = time.perf_counter()
        q_latencies_ms.append((t1 - t0) * 1000)

q_latencies_ms = np.array(q_latencies_ms)

print("\n" + "=" * 60)
print("INT8 QUANTIZED CPU LATENCY (batch=1, single-threaded)")
print("=" * 60)
print(f"Mean    : {q_latencies_ms.mean():>8.2f} ms   (FP32: {latencies_ms.mean():.2f})")
print(f"Median  : {np.median(q_latencies_ms):>8.2f} ms   (FP32: {np.median(latencies_ms):.2f})")
print(f"Std     : {q_latencies_ms.std():>8.2f} ms")
print(f"p95     : {np.percentile(q_latencies_ms, 95):>8.2f} ms")
print(f"p99     : {np.percentile(q_latencies_ms, 99):>8.2f} ms")
print("=" * 60)

# Speedup
speedup_median = np.median(latencies_ms) / np.median(q_latencies_ms)
speedup_mean   = latencies_ms.mean() / q_latencies_ms.mean()
print(f"\nSpeedup (median): {speedup_median:.2f}x")
print(f"Speedup (mean)  : {speedup_mean:.2f}x")

In [ ]:
# Phase 3b: Quantized sensitivity + specificity
# Same holdout, same healthy set, same preprocessing.

# BraTS holdout — seed 2026
q_sens, q_s_correct, q_s_total, q_brats_preds = eval_metric(
    quantized_model, BRATS_SOURCE_DIR, holdout_files, 'tumor'
)
print(f"Sensitivity: {q_sens:.4f}  ({q_s_correct}/{q_s_total})   [FP32: {sens:.4f}]")

# Nigerian healthy
q_spec, q_h_correct, q_h_total, q_healthy_preds = eval_metric(
    quantized_model, NIGERIAN_HEALTHY, healthy_files, 'healthy'
)
print(f"Specificity: {q_spec:.4f}  ({q_h_correct}/{q_h_total})   [FP32: {spec:.4f}]")

print("\n" + "=" * 60)
print("INT8 QUANTIZED CLINICAL METRICS (CPU)")
print("=" * 60)
print(f"Sensitivity : {q_sens:.2%}  ({q_s_correct}/{q_s_total})   Δ from FP32: {(q_sens-sens)*100:+.2f} pp")
print(f"Specificity : {q_spec:.2%}  ({q_h_correct}/{q_h_total})   Δ from FP32: {(q_spec-spec)*100:+.2f} pp")
print("=" * 60)

In [ ]:
# Phase 4: Persist quantized results + apply decision gate.

QUANTIZED_JSON = os.path.join(QUANT_EXP_DIR, "quantized_int8.json")

quantized_results = {
    'phase'                 : 'INT8 dynamic quantization',
    'source_model'          : 'FP32 baseline (seed 2026)',
    'device'                : str(device),
    'num_threads'           : torch.get_num_threads(),

    # Size
    'disk_state_dict_mb'    : round(state_dict_size_mb, 2),
    'compression_ratio'     : round(disk_size_mb / state_dict_size_mb, 2),
    'quantized_linear_count': 50,

    # Latency
    'latency_mean_ms'       : round(float(q_latencies_ms.mean()), 2),
    'latency_median_ms'     : round(float(np.median(q_latencies_ms)), 2),
    'latency_std_ms'        : round(float(q_latencies_ms.std()), 2),
    'latency_p95_ms'        : round(float(np.percentile(q_latencies_ms, 95)), 2),
    'latency_p99_ms'        : round(float(np.percentile(q_latencies_ms, 99)), 2),
    'speedup_median'        : round(float(speedup_median), 2),
    'speedup_mean'          : round(float(speedup_mean), 2),

    # Clinical
    'sensitivity'           : round(q_sens, 4),
    'sensitivity_delta_pp'  : round((q_sens - sens) * 100, 2),
    'specificity'           : round(q_spec, 4),
    'specificity_delta_pp'  : round((q_spec - spec) * 100, 2),
}

with open(QUANTIZED_JSON, 'w') as f:
    json.dump(quantized_results, f, indent=2)

# --- Decision gate (defined BEFORE we saw the numbers) ---
ACCEPT_THRESHOLD_PP = 3.0   # ≤3 percentage points drop = accept

sens_drop = (sens - q_sens) * 100
spec_drop = (spec - q_spec) * 100

print("\n" + "=" * 70)
print(f"{'PHASE 4 — DECISION GATE':^70}")
print("=" * 70)
print(f"Acceptance threshold: sensitivity AND specificity drop ≤ {ACCEPT_THRESHOLD_PP} pp")
print("-" * 70)
print(f"Sensitivity drop : {sens_drop:+.2f} pp   {' PASS' if sens_drop <= ACCEPT_THRESHOLD_PP else ' FAIL'}")
print(f"Specificity drop : {spec_drop:+.2f} pp   {' PASS' if spec_drop <= ACCEPT_THRESHOLD_PP else ' FAIL'}")

if sens_drop <= ACCEPT_THRESHOLD_PP and spec_drop <= ACCEPT_THRESHOLD_PP:
    verdict = " ACCEPT — quantization preserves clinical metrics"
else:
    verdict = " REJECT — degradation exceeds threshold; motivates QAT or TinyViT"print(f"\nVERDICT: {verdict}")
print("=" * 70)
print(f"\nResults saved: {QUANTIZED_JSON}")

In [ ]:
# Which specific scan(s) flipped between FP32 and INT8?

# You need the FP32 healthy_preds from Phase 1 — if you don't
# still have it in memory, re-run one line:
fp32_spec, _, _, fp32_healthy_preds = eval_metric(
    model, NIGERIAN_HEALTHY, healthy_files, 'healthy'
)

print(f"{'Filename':<25} {'FP32 prob':>10} {'INT8 prob':>10} {'FP32':>7} {'INT8':>7} {'Flipped?':>10}")
print("-" * 75)
for i, f in enumerate(healthy_files):
    fp32_p  = fp32_healthy_preds[i]
    int8_p  = q_healthy_preds[i]
    fp32_cls = 'tumor' if fp32_p > 0.5 else 'healthy'
    int8_cls = 'tumor' if int8_p > 0.5 else 'healthy'
    flipped = ' YES' if fp32_cls != int8_cls else ''
    if flipped or abs(fp32_p - int8_p) > 0.05:  # show flips + notable drift
        print(f"{f:<25} {fp32_p:>10.4f} {int8_p:>10.4f} {fp32_cls:>7} {int8_cls:>7} {flipped:>10}")

In [ ]:
# Side-by-side histograms: FP32 vs INT8 predictions
# Top row : BraTS holdout (tumor ground truth  want probs near 1)
# Bottom  : Nigerian healthy (healthy ground truth  want probs near 0)

# Make sure we have FP32 predictions on both sets.
# If you already ran the flipped-scan cell, fp32_healthy_preds exists.
# Recompute BraTS FP32 preds if not in memory:
import matplotlib.pyplot as plt

try:
    fp32_brats_preds
except NameError:
    _, _, _, fp32_brats_preds = eval_metric(
        model, BRATS_SOURCE_DIR, holdout_files, 'tumor'
    )

try:
    fp32_healthy_preds
except NameError:
    _, _, _, fp32_healthy_preds = eval_metric(
        model, NIGERIAN_HEALTHY, healthy_files, 'healthy'
    )

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# --- Row 1: BraTS holdout ---
axes[0, 0].hist(fp32_brats_preds, bins=20, range=(0, 1),
                color='teal', edgecolor='black', alpha=0.85)
axes[0, 0].axvline(0.5, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].set_title(f"FP32 — BraTS holdout  (sens {sens:.2%})", fontsize=11)
axes[0, 0].set_xlabel("Tumor probability")
axes[0, 0].set_ylabel("Count")

axes[0, 1].hist(q_brats_preds, bins=20, range=(0, 1),
                color='darkcyan', edgecolor='black', alpha=0.85)
axes[0, 1].axvline(0.5, color='red', linestyle='--', linewidth=1.5)
axes[0, 1].set_title(f"INT8 — BraTS holdout  (sens {q_sens:.2%})", fontsize=11)
axes[0, 1].set_xlabel("Tumor probability")

# --- Row 2: Nigerian healthy ---
axes[1, 0].hist(fp32_healthy_preds, bins=20, range=(0, 1),
                color='steelblue', edgecolor='black', alpha=0.85)
axes[1, 0].axvline(0.5, color='red', linestyle='--', linewidth=1.5)
axes[1, 0].set_title(f"FP32 — Nigerian healthy  (spec {spec:.2%})", fontsize=11)
axes[1, 0].set_xlabel("Tumor probability")
axes[1, 0].set_ylabel("Count")

axes[1, 1].hist(q_healthy_preds, bins=20, range=(0, 1),
                color='navy', edgecolor='black', alpha=0.85)
axes[1, 1].axvline(0.5, color='red', linestyle='--', linewidth=1.5)
axes[1, 1].set_title(f"INT8 — Nigerian healthy  (spec {q_spec:.2%})", fontsize=11)
axes[1, 1].set_xlabel("Tumor probability")

plt.suptitle("FP32 vs INT8 Dynamic Quantization — Prediction Distributions",
             fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

# --- Bonus: scatter plot showing per-scan drift ---
# Each point = one scan. x = FP32 prob, y = INT8 prob.
# Points on the diagonal = no drift. Off-diagonal = quantization moved that scan's score.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(fp32_brats_preds, q_brats_preds, alpha=0.6, s=40, color='teal', edgecolor='black')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='no drift')
axes[0].axvline(0.5, color='red', linestyle=':', alpha=0.5)
axes[0].axhline(0.5, color='red', linestyle=':', alpha=0.5)
axes[0].set_xlabel("FP32 tumor probability")
axes[0].set_ylabel("INT8 tumor probability")
axes[0].set_title(f"BraTS holdout — per-scan drift (n={len(fp32_brats_preds)})")
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].scatter(fp32_healthy_preds, q_healthy_preds, alpha=0.6, s=40, color='steelblue', edgecolor='black')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='no drift')
axes[1].axvline(0.5, color='red', linestyle=':', alpha=0.5)
axes[1].axhline(0.5, color='red', linestyle=':', alpha=0.5)
axes[1].set_xlabel("FP32 tumor probability")
axes[1].set_ylabel("INT8 tumor probability")
axes[1].set_title(f"Nigerian healthy — per-scan drift (n={len(fp32_healthy_preds)})")
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
axes[1].legend()

plt.suptitle("Per-scan drift: FP32 vs INT8 predictions", fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

# --- Quantitative drift summary ---
brats_drift   = np.abs(np.array(q_brats_preds)   - np.array(fp32_brats_preds))
healthy_drift = np.abs(np.array(q_healthy_preds) - np.array(fp32_healthy_preds))

print("=" * 60)
print("PREDICTION DRIFT: |INT8 - FP32|")
print("=" * 60)
print(f"BraTS holdout   — mean {brats_drift.mean():.4f}, max {brats_drift.max():.4f}")
print(f"Nigerian healthy — mean {healthy_drift.mean():.4f}, max {healthy_drift.max():.4f}")
print("=" * 60)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

                if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

# --- Pick one patient, show all 5 slices ---
by_patient = defaultdict(list)
for f in files:
    pid = f.split('_slice')[0]
    by_patient[pid].append(f)

pid = list(by_patient.keys())[0]
slices = sorted(by_patient[pid])
print(f"Patient {pid} — {len(slices)} files:")
for s in slices:
    print(f"{s}")

# --- Visual: display all 5, plus their pixel-value statistics ---
fig, axes = plt.subplots(1, len(slices), figsize=(4*len(slices), 4.5))
for ax, fname in zip(axes, slices):
    img = cv2.imread(os.path.join(BRATS_SOURCE, fname), cv2.IMREAD_GRAYSCALE)
    unique_vals = len(np.unique(img))
    nonzero_pct = 100 * np.count_nonzero(img) / img.size
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{fname.split('_')[-1]}\n"f"unique px vals: {unique_vals}\n"f"non-zero: {nonzero_pct:.1f}%",
                 fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()